# 06_compare_district_area_with_official_stats

**Role.** District-level agreement with official statistics and spatial consistency diagnostics.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [1]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os, sys, json, warnings
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest, append_manifest_note
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "05_compare_district_area_with_official_stats.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


Project root: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak
Notebook: 05_compare_district_area_with_official_stats.ipynb
Classes: 10 | Coffee classes: [1, 2, 3] | Random seed: 2024


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import (
    detect_column, detect_label_columns, assert_class_ids,
    class_count_table, warn_if_balanced, extract_probability_columns,
    assert_probability_matrix,
)
from coffeemap.metrics import classification_summary, overall_metrics, shannon_entropy, probability_margin
from coffeemap.validation import audit_validation_predictions
from coffeemap.olofsson import error_matrix_counts, area_adjustment, binary_coffee_area_adjustment

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


## A. Figure 8: District-level area validation


In [3]:
# -*- coding: utf-8 -*-
"""
08, Figure 8: District-level area consistency of mapped coffee extent
Dak Lak Province, Vietnam, 2024

Professional update
-------------------
This script reads three inputs:
  1) Official district statistics:
     daklak_coffee_mapping_validation_2024.csv
  2) GEE mapped coffee area by district:
     GEE_DakLak_Coffee_Area_By_District_2024.csv
  3) GEE per-class area statistics:
     Table_AreaStatistics_DakLak2024.csv

It then:
  - normalizes Vietnamese district names,
  - merges official statistics with GEE mapped area,
  - audits area consistency between district aggregation and class aggregation,
  - computes area-level agreement metrics,
  - exports a cleaned validation table and audit report,
  - generates publication-ready Figure 8 and supplementary diagnostics.

Notes
-----
This is an external area-level consistency assessment. It is not pixel-level
classification accuracy.
"""

from __future__ import annotations

import os
import re
import unicodedata
import warnings
from pathlib import Path
from typing import Iterable

import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text
from matplotlib import font_manager
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from scipy.stats import linregress, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =============================================================================
# 1. SETTINGS
# =============================================================================

# Input file names. The script searches in candidate folders below.
OFFICIAL_STATS_FILE = "daklak_coffee_mapping_validation_2024.csv"  # place this under ./data/
GEE_DISTRICT_AREA_FILE = "GEE_DakLak_Coffee_Area_By_District_2024.csv"
GEE_CLASS_AREA_FILE = "Table_AreaStatistics_DakLak2024.csv"

SEARCH_DIRS = [
    Path("data/raw"),
    Path("data/raw/data_DakLak_Statistics"),
    Path("."),
    Path("data_DakLak_Statistics"),
    Path("GEE_Exports_R3000"),
    Path("data"),
    Path("/mnt/data"),
]

# output dirs inherited from bootstrap: FIGURES_DIR, TABLES_DIR, SUPPLEMENTARY_DIR
OUT_DIR = FIGURES_DIR

# Column names in official statistics file
DISTRICT_EN_COL = "district_name_en"
DISTRICT_VI_COL = "district_name"
OFFICIAL_COL = "planted_area_ha"
HARVESTED_COL = "harvested_area_ha"
MAPPED_COL = "mapped_coffee_area_ha"

# Column names in GEE district-area file
GEE_DISTRICT_COL = "district_name_raw"
GEE_MAPPED_COL = "mapped_coffee_area_ha"
GEE_SUN_COL = "mapped_sun_coffee_area_ha"
GEE_INTERCROP_COL = "mapped_intercrop_coffee_area_ha"
GEE_YOUNG_COL = "mapped_newly_planted_coffee_area_ha"
GEE_CHECK_COL = "mapped_coffee_area_check_ha"

# Bootstrap settings
N_BOOT = int(os.environ.get("FIG8_N_BOOT", "10000"))
if N_BOOT < 1:
    raise ValueError("FIG8_N_BOOT must be >= 1.")
BOOT_SEED = 2024

DPI = 600
REQUESTED_FONT = os.environ.get("FIG8_FONT", "Arial")
_AVAILABLE_FONTS = {f.name for f in font_manager.fontManager.ttflist}
FONT_FAMILY = REQUESTED_FONT if REQUESTED_FONT in _AVAILABLE_FONTS else "DejaVu Sans"
if FONT_FAMILY != REQUESTED_FONT:
    warnings.warn(
        f"Requested font '{REQUESTED_FONT}' was not found. Falling back to '{FONT_FAMILY}'.",
        RuntimeWarning,
    )

SAVE_FORMATS = os.environ.get("FIG8_SAVE_FORMATS", "png,pdf,svg").split(",")
SAVE_FORMATS = [x.strip().lower() for x in SAVE_FORMATS if x.strip()]

# Colors
COLOR_POINT = "#2C6F8E"
COLOR_FIT = "#B23A48"
COLOR_ONE_TO_ONE = "#4D4D4D"
COLOR_UNDER = "#2C6F8E"
COLOR_OVER = "#B23A48"
COLOR_NEUTRAL = "#555555"
COLOR_TEXT = "#222222"

PANEL_LABEL_X = -0.10
PANEL_LABEL_Y = 1.10
PANEL_LABEL_FONTSIZE = 11
PANEL_LABEL_X_BA = -0.06
PANEL_LABEL_Y_BA = 1.08


# =============================================================================
# 2. STYLE
# =============================================================================

plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": 12,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)


# =============================================================================
# 3. FILE AND NAME HELPERS
# =============================================================================


def find_file(filename: str, search_dirs: Iterable[Path] = SEARCH_DIRS) -> Path:
    """Find a file from common project folders."""
    for d in search_dirs:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n"
        + "\n".join(str(d / filename) for d in search_dirs)
    )


def normalize_district_name(name: str) -> str:
    """
    Normalize Vietnamese district names for robust joins:
      - lower case
      - remove accents
      - convert đ -> d
      - remove punctuation and spaces
      - normalize common aliases, e.g. Thị Xã Buôn Hồ -> BuonHo
    """
    if pd.isna(name):
        return ""

    s = str(name).strip().lower()
    s = s.replace("đ", "d")
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = re.sub(r"[^a-z0-9]+", "", s)

    aliases = {
        "thixabuonho": "buonho",
        "txbuonho": "buonho",
        "krongan": "krongana",
        "krongana": "krongana",
        "lak": "lak",
        "lac": "lak",
        "mdrak": "mdrak",
        "mdrac": "mdrak",
        "cumgar": "cumgar",
        "cugar": "cumgar",
    }
    return aliases.get(s, s)


def read_csv_clean(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig")


# =============================================================================
# 4. FORMAT HELPERS
# =============================================================================


def axis_kha(x, pos):
    """Format hectares as thousand hectares on axes."""
    return f"{x / 1000:.0f}"


def fmt_ha(x):
    return "NA" if pd.isna(x) else f"{x:,.0f}"


def signed_ha(x):
    return "NA" if pd.isna(x) else f"{x:+,.0f}"


def signed_pct(x):
    return "NA" if pd.isna(x) else f"{x:+.1f}%"


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.tick_params(direction="out", length=3, width=0.6)


def add_panel_label(ax, label, x=PANEL_LABEL_X, y=PANEL_LABEL_Y):
    if label is None:
        return
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=PANEL_LABEL_FONTSIZE,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False,
    )


# =============================================================================
# 5. LOAD AND MERGE DATA
# =============================================================================

official_path = find_file(OFFICIAL_STATS_FILE)
gee_district_path = find_file(GEE_DISTRICT_AREA_FILE)
gee_class_path = find_file(GEE_CLASS_AREA_FILE)

official = read_csv_clean(official_path)
gee_district = read_csv_clean(gee_district_path)
gee_class = read_csv_clean(gee_class_path)

# Basic column checks
official_required = [DISTRICT_EN_COL, DISTRICT_VI_COL, OFFICIAL_COL]
gee_required = [
    GEE_DISTRICT_COL,
    GEE_MAPPED_COL,
    GEE_SUN_COL,
    GEE_INTERCROP_COL,
    GEE_YOUNG_COL,
]
missing_off = [c for c in official_required if c not in official.columns]
missing_gee = [c for c in gee_required if c not in gee_district.columns]
if missing_off:
    raise ValueError(f"Official file missing columns: {missing_off}")
if missing_gee:
    raise ValueError(f"GEE district file missing columns: {missing_gee}")

official["_join_key"] = official[DISTRICT_VI_COL].apply(normalize_district_name)
gee_district["_join_key"] = gee_district[GEE_DISTRICT_COL].apply(
    normalize_district_name
)

if official["_join_key"].duplicated().any():
    dup = official.loc[
        official["_join_key"].duplicated(keep=False), [DISTRICT_VI_COL, "_join_key"]
    ]
    raise ValueError(f"Duplicated official district join keys:\n{dup}")

if gee_district["_join_key"].duplicated().any():
    dup = gee_district.loc[
        gee_district["_join_key"].duplicated(keep=False),
        [GEE_DISTRICT_COL, "_join_key"],
    ]
    raise ValueError(f"Duplicated GEE district join keys:\n{dup}")

gee_merge = gee_district[
    [
        "_join_key",
        GEE_DISTRICT_COL,
        GEE_MAPPED_COL,
        GEE_SUN_COL,
        GEE_INTERCROP_COL,
        GEE_YOUNG_COL,
    ]
    + ([GEE_CHECK_COL] if GEE_CHECK_COL in gee_district.columns else [])
].copy()

# Rename GEE columns before merge to avoid collisions with placeholder columns
# already present in the official template, e.g. mapped_coffee_area_ha.
gee_rename = {
    GEE_DISTRICT_COL: "district_name_gee",
    GEE_MAPPED_COL: "gee_mapped_coffee_area_ha",
    GEE_SUN_COL: "gee_mapped_sun_coffee_area_ha",
    GEE_INTERCROP_COL: "gee_mapped_intercrop_coffee_area_ha",
    GEE_YOUNG_COL: "gee_mapped_newly_planted_coffee_area_ha",
}
if GEE_CHECK_COL in gee_merge.columns:
    gee_rename[GEE_CHECK_COL] = "gee_mapped_coffee_area_check_ha"

gee_merge = gee_merge.rename(columns=gee_rename)

merged = official.merge(
    gee_merge,
    on="_join_key",
    how="left",
    validate="one_to_one",
)

unmatched = merged[merged["gee_mapped_coffee_area_ha"].isna()].copy()
if not unmatched.empty:
    raise ValueError(
        "Some official districts were not matched to GEE district-area rows:\n"
        + unmatched[[DISTRICT_VI_COL, DISTRICT_EN_COL, "_join_key"]].to_string(
            index=False
        )
    )

# Numeric conversions
numeric_cols = [
    OFFICIAL_COL,
    HARVESTED_COL,
    "gee_mapped_coffee_area_ha",
    "gee_mapped_sun_coffee_area_ha",
    "gee_mapped_intercrop_coffee_area_ha",
    "gee_mapped_newly_planted_coffee_area_ha",
]
for col in numeric_cols:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# Fill final mapped fields in official validation table
merged[MAPPED_COL] = merged["gee_mapped_coffee_area_ha"]
merged["mapped_sun_coffee_area_ha"] = merged["gee_mapped_sun_coffee_area_ha"]
merged["mapped_intercrop_coffee_area_ha"] = merged[
    "gee_mapped_intercrop_coffee_area_ha"
]
merged["mapped_newly_planted_coffee_area_ha"] = merged[
    "gee_mapped_newly_planted_coffee_area_ha"
]

# Consistency check: component sum
merged["mapped_coffee_area_check_ha"] = (
    merged["mapped_sun_coffee_area_ha"]
    + merged["mapped_intercrop_coffee_area_ha"]
    + merged["mapped_newly_planted_coffee_area_ha"]
)
merged["component_sum_difference_ha"] = (
    merged[MAPPED_COL] - merged["mapped_coffee_area_check_ha"]
)

# Residual columns
merged["difference_ha"] = merged[MAPPED_COL] - merged[OFFICIAL_COL]
merged["relative_difference_percent"] = np.where(
    merged[OFFICIAL_COL] > 0,
    merged["difference_ha"] / merged[OFFICIAL_COL] * 100,
    np.nan,
)
merged["absolute_error_ha"] = merged["difference_ha"].abs()
merged["absolute_percentage_error"] = merged["relative_difference_percent"].abs()
merged["mean_area_ha"] = (merged[MAPPED_COL] + merged[OFFICIAL_COL]) / 2

# Sort for outputs
valid = merged.copy()
valid_scatter = valid.copy()
valid_resid = valid.sort_values("difference_ha", ascending=True).copy()


# =============================================================================
# 6. AREA AUDIT
# =============================================================================

district_total = float(valid[MAPPED_COL].sum())
district_sun = float(valid["mapped_sun_coffee_area_ha"].sum())
district_intercrop = float(valid["mapped_intercrop_coffee_area_ha"].sum())
district_young = float(valid["mapped_newly_planted_coffee_area_ha"].sum())

# Per-class totals from Table_AreaStatistics
class_coffee = gee_class[gee_class["class_id"].isin([1, 2, 3])].copy()
class_total = float(class_coffee["area_ha"].sum())
class_by_id = dict(
    zip(class_coffee["class_id"].astype(int), class_coffee["area_ha"].astype(float))
)

area_audit = pd.DataFrame(
    [
        {
            "check": "Total coffee: district aggregation vs per-class table",
            "district_aggregation_ha": district_total,
            "per_class_table_ha": class_total,
            "difference_ha": district_total - class_total,
            "relative_difference_percent": (district_total - class_total)
            / class_total
            * 100
            if class_total != 0
            else np.nan,
        },
        {
            "check": "Sun coffee",
            "district_aggregation_ha": district_sun,
            "per_class_table_ha": class_by_id.get(1, np.nan),
            "difference_ha": district_sun - class_by_id.get(1, np.nan),
            "relative_difference_percent": (district_sun - class_by_id.get(1, np.nan))
            / class_by_id.get(1, np.nan)
            * 100
            if class_by_id.get(1, np.nan)
            else np.nan,
        },
        {
            "check": "Intercrop coffee",
            "district_aggregation_ha": district_intercrop,
            "per_class_table_ha": class_by_id.get(2, np.nan),
            "difference_ha": district_intercrop - class_by_id.get(2, np.nan),
            "relative_difference_percent": (
                district_intercrop - class_by_id.get(2, np.nan)
            )
            / class_by_id.get(2, np.nan)
            * 100
            if class_by_id.get(2, np.nan)
            else np.nan,
        },
        {
            "check": "Newly planted coffee",
            "district_aggregation_ha": district_young,
            "per_class_table_ha": class_by_id.get(3, np.nan),
            "difference_ha": district_young - class_by_id.get(3, np.nan),
            "relative_difference_percent": (district_young - class_by_id.get(3, np.nan))
            / class_by_id.get(3, np.nan)
            * 100
            if class_by_id.get(3, np.nan)
            else np.nan,
        },
    ]
)

# Warn if totals differ more than 0.5% or 100 ha
total_diff = abs(area_audit.loc[0, "difference_ha"])
total_rel = abs(area_audit.loc[0, "relative_difference_percent"])
if total_diff > 100 and total_rel > 0.5:
    warnings.warn(
        "District coffee total differs substantially from per-class coffee total. "
        "Check classified asset, AOI, and district boundaries.",
        RuntimeWarning,
    )


# =============================================================================
# 7. METRICS
# =============================================================================


def compute_metrics(df: pd.DataFrame, label: str) -> dict:
    y_true = df[OFFICIAL_COL].to_numpy(dtype=float)
    y_pred = df[MAPPED_COL].to_numpy(dtype=float)

    official_total = float(y_true.sum())
    mapped_total = float(y_pred.sum())
    bias_ha = mapped_total - official_total
    relative_bias_percent = (
        bias_ha / official_total * 100 if official_total != 0 else np.nan
    )

    mae_ha = float(mean_absolute_error(y_true, y_pred))
    rmse_ha = float(np.sqrt(mean_squared_error(y_true, y_pred)))

    if len(df) >= 2 and np.std(y_true) > 0:
        r2 = float(r2_score(y_true, y_pred))
    else:
        r2 = np.nan

    nonzero = df[df[OFFICIAL_COL] > 0].copy()
    mape_percent = (
        float((nonzero["absolute_error_ha"] / nonzero[OFFICIAL_COL]).mean() * 100)
        if len(nonzero)
        else np.nan
    )

    if len(df) >= 3 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        pearson_r, pearson_p = pearsonr(y_true, y_pred)
        reg = linregress(y_true, y_pred)
    else:
        pearson_r, pearson_p, reg = np.nan, np.nan, None

    diff = df["difference_ha"].to_numpy(dtype=float)
    mean_diff = float(np.mean(diff))
    sd_diff = float(np.std(diff, ddof=1)) if len(diff) > 1 else np.nan
    loa_lower = mean_diff - 1.96 * sd_diff if np.isfinite(sd_diff) else np.nan
    loa_upper = mean_diff + 1.96 * sd_diff if np.isfinite(sd_diff) else np.nan

    return {
        "dataset": label,
        "n_districts": int(len(df)),
        "n_nonzero_official_area": int((df[OFFICIAL_COL] > 0).sum()),
        "official_total_area_ha": official_total,
        "mapped_total_area_ha": mapped_total,
        "bias_ha": bias_ha,
        "relative_bias_percent": relative_bias_percent,
        "MAE_ha": mae_ha,
        "RMSE_ha": rmse_ha,
        "MAPE_percent_nonzero_official": mape_percent,
        "Pearson_r": float(pearson_r) if not pd.isna(pearson_r) else np.nan,
        "Pearson_p": float(pearson_p) if not pd.isna(pearson_p) else np.nan,
        "R2": r2,
        "regression_slope": float(reg.slope) if reg is not None else np.nan,
        "regression_intercept_ha": float(reg.intercept) if reg is not None else np.nan,
        "mean_difference_ha": mean_diff,
        "sd_difference_ha": sd_diff,
        "loa_lower_ha": loa_lower,
        "loa_upper_ha": loa_upper,
    }


metrics_all = compute_metrics(valid, "all_districts")
valid_nonzero = valid[valid[OFFICIAL_COL] > 0].copy()
metrics_nonzero = compute_metrics(valid_nonzero, "nonzero_official_area_only")

metrics = pd.DataFrame([metrics_all, metrics_nonzero])


def bootstrap_metrics(df: pd.DataFrame, n_boot=N_BOOT, seed=BOOT_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(df)
    rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        m = compute_metrics(df.iloc[idx].copy(), label="bootstrap")
        rows.append(
            {
                "bias_ha": m["bias_ha"],
                "relative_bias_percent": m["relative_bias_percent"],
                "MAE_ha": m["MAE_ha"],
                "RMSE_ha": m["RMSE_ha"],
                "MAPE_percent_nonzero_official": m["MAPE_percent_nonzero_official"],
                "Pearson_r": m["Pearson_r"],
                "R2": m["R2"],
                "regression_slope": m["regression_slope"],
            }
        )
    return pd.DataFrame(rows)


def summarize_bootstrap(boot: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in boot.columns:
        vals = boot[col].replace([np.inf, -np.inf], np.nan).dropna()
        rows.append(
            {
                "metric": col,
                "estimate_boot_mean": float(vals.mean()) if len(vals) else np.nan,
                "ci95_low": float(vals.quantile(0.025)) if len(vals) else np.nan,
                "ci95_high": float(vals.quantile(0.975)) if len(vals) else np.nan,
            }
        )
    return pd.DataFrame(rows)


boot = bootstrap_metrics(valid)
boot_summary = summarize_bootstrap(boot)


# Use all-district metrics for main figure and totals
bias_ha = metrics_all["bias_ha"]
relative_bias_percent = metrics_all["relative_bias_percent"]
mae_ha = metrics_all["MAE_ha"]
rmse_ha = metrics_all["RMSE_ha"]
mape_percent = metrics_all["MAPE_percent_nonzero_official"]
pearson_r = metrics_all["Pearson_r"]
r2 = metrics_all["R2"]
slope = metrics_all["regression_slope"]
intercept = metrics_all["regression_intercept_ha"]
mean_diff = metrics_all["mean_difference_ha"]
loa_lower = metrics_all["loa_lower_ha"]
loa_upper = metrics_all["loa_upper_ha"]

bias_direction = (
    "positive" if bias_ha > 0 else "negative" if bias_ha < 0 else "near-zero"
)
bias_phrase = (
    "positive bias / overestimation"
    if bias_ha > 0
    else "negative bias / underestimation"
    if bias_ha < 0
    else "near-zero total bias"
)


# =============================================================================
# 8. EXPORT CLEANED DATA AND AUDITS
# =============================================================================

ordered_cols = [
    DISTRICT_EN_COL,
    DISTRICT_VI_COL,
    "district_name_gee",
    "year",
    OFFICIAL_COL,
    HARVESTED_COL,
    "production_ton",
    "yield_ton_per_ha",
    MAPPED_COL,
    "mapped_sun_coffee_area_ha",
    "mapped_intercrop_coffee_area_ha",
    "mapped_newly_planted_coffee_area_ha",
    "difference_ha",
    "relative_difference_percent",
    "absolute_error_ha",
    "absolute_percentage_error",
    "mean_area_ha",
    "_join_key",
]
ordered_cols = [c for c in ordered_cols if c in valid.columns]

area_audit.to_csv(
    SUPPLEMENTARY_DIR / "DistrictVsClassArea_Audit_20260715.csv",
    index=False,
    encoding="utf-8-sig",
)

audit_text = f"""Figure 8 data audit
===================

Official statistics file:
  {official_path}

GEE district-area file:
  {gee_district_path}

GEE class-area file:
  {gee_class_path}

Matched districts:
  {len(valid)} / {len(official)}

Unmatched districts:
  none

District-total mapped coffee area:
  {district_total:,.2f} ha

Per-class-table mapped coffee area:
  {class_total:,.2f} ha

Difference:
  {district_total - class_total:+,.2f} ha
  {(district_total - class_total) / class_total * 100:+.4f} %

Official planted coffee area:
  {metrics_all["official_total_area_ha"]:,.2f} ha

Mapped coffee area:
  {metrics_all["mapped_total_area_ha"]:,.2f} ha

Total bias:
  {metrics_all["bias_ha"]:+,.2f} ha
  {metrics_all["relative_bias_percent"]:+.2f} %

Interpretation:
  {bias_phrase}

Important note:
  This is an area-level consistency assessment, not pixel-level accuracy.
"""
(SUPPLEMENTARY_DIR / "Figure9_Data_Audit_Report.txt").write_text(audit_text, encoding="utf-8")


# =============================================================================
# 9. AXIS LIMITS
# =============================================================================

max_area = max(valid[OFFICIAL_COL].max(), valid[MAPPED_COL].max())
max_val = np.ceil(max_area / 5000) * 5000
if max_val <= 0:
    max_val = max_area * 1.1

max_abs_resid = max(valid_resid["difference_ha"].abs().max(), 1)
resid_lim = np.ceil(max_abs_resid / 1000) * 1000
resid_xlim = (-resid_lim * 1.15, resid_lim * 1.15)

ba_y_min = min(valid["difference_ha"].min(), loa_lower, 0) * 1.15
ba_y_max = max(valid["difference_ha"].max(), loa_upper, 0) * 1.15
if ba_y_max == 0:
    ba_y_max = abs(ba_y_min) * 0.18
ba_ylim = (ba_y_min, ba_y_max)


# =============================================================================
# 10. PLOTTING FUNCTIONS
# =============================================================================


def residual_label(row):
    if row[OFFICIAL_COL] == 0:
        return f"+{row[MAPPED_COL] / 1000:.1f}k ha"
    return f"{row['relative_difference_percent']:+.1f}%"


def draw_scatter(ax, panel_label=None):
    ax.scatter(
        valid_scatter[OFFICIAL_COL],
        valid_scatter[MAPPED_COL],
        s=42,
        facecolor=COLOR_POINT,
        edgecolor="white",
        linewidth=0.45,
        alpha=0.93,
        zorder=3,
    )

    ax.plot(
        [0, max_val],
        [0, max_val],
        color=COLOR_ONE_TO_ONE,
        linestyle=(0, (4, 2)),
        linewidth=0.9,
        label="1:1 line",
        zorder=1,
    )

    if np.isfinite(slope) and np.isfinite(intercept):
        x_line = np.linspace(0, max_val, 100)
        y_line = intercept + slope * x_line
        ax.plot(
            x_line,
            y_line,
            color=COLOR_FIT,
            linewidth=1.2,
            label="Linear regression",
            zorder=2,
        )
        eq_text = f"y = {slope:.3f}x {intercept:+,.0f}"
    else:
        eq_text = "Linear regression: n.a."

    ax.set_xlim(0, max_val)
    ax.set_ylim(0, max_val)
    ax.xaxis.set_major_formatter(FuncFormatter(axis_kha))
    ax.yaxis.set_major_formatter(FuncFormatter(axis_kha))
    ax.set_xlabel("Official planted coffee area (10³ ha)")
    ax.set_ylabel("Mapped coffee area (10³ ha)")

    metric_text = (
        f"$R^2$ = {r2:.3f}\n"
        f"Pearson $r$ = {pearson_r:.3f}\n"
        f"RMSE = {rmse_ha / 1000:.1f}k ha\n"
        f"MAE = {mae_ha / 1000:.1f}k ha\n"
        f"Bias = {bias_ha / 1000:+.1f}k ha ({relative_bias_percent:+.1f}%)\n"
        f"{eq_text}"
    )

    metric_box = ax.text(
        0.045,
        0.955,
        metric_text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=6.7,
        linespacing=1.22,
        bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="#D0D0D0", lw=0.6),
    )


    label_df = valid_scatter  # label all districts; adjustText below prevents collisions
    texts_scatter = [
        ax.text(row[OFFICIAL_COL], row[MAPPED_COL], str(row[DISTRICT_EN_COL]),
                fontsize=6.6, color=COLOR_TEXT)
        for _, row in label_df.iterrows()
    ]
    adjust_text(
        texts_scatter, ax=ax,
        x=label_df[OFFICIAL_COL].to_numpy(), y=label_df[MAPPED_COL].to_numpy(),
        objects=[metric_box],
        arrowprops=dict(arrowstyle="-", color="#999999", lw=0.4),
        force_text=(0.8, 1.0), force_static=(0.4, 0.5), expand=(1.3, 1.6),
        max_move=(40, 40), time_lim=8,
    )
    style_axes(ax)
    add_panel_label(ax, panel_label)


def draw_residuals(ax, panel_label=None):
    y = np.arange(len(valid_resid))
    vals = valid_resid["difference_ha"].to_numpy()
    colors = np.where(vals >= 0, COLOR_OVER, COLOR_UNDER)

    ax.barh(y, vals, color=colors, edgecolor="none", height=0.72)
    ax.axvline(0, color="#333333", linewidth=0.75)

    ax.set_yticks(y)
    ax.set_yticklabels(valid_resid[DISTRICT_EN_COL].astype(str).values)
    ax.set_xlim(resid_xlim)
    ax.xaxis.set_major_formatter(FuncFormatter(axis_kha))
    ax.set_xlabel("Mapped − official planted area (10³ ha)")

    offset = max_abs_resid * 0.035
    for i, (_, row) in enumerate(valid_resid.iterrows()):
        lab = residual_label(row)
        val = row["difference_ha"]
        x = val + (offset if val >= 0 else -offset)
        ha = "left" if val >= 0 else "right"
        ax.text(x, i, lab, va="center", ha=ha, fontsize=6.3, color=COLOR_TEXT)

    ax.text(
        0.02,
        0.985,
        "Underestimation",
        color=COLOR_UNDER,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=7.1,
    )
    ax.text(
        0.98,
        0.985,
        "Overestimation",
        color=COLOR_OVER,
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=7.1,
    )

    style_axes(ax)
    add_panel_label(ax, panel_label)


def draw_bland_altman(ax, panel_label=None):
    ax.scatter(
        valid["mean_area_ha"],
        valid["difference_ha"],
        s=30,
        facecolor=COLOR_POINT,
        edgecolor="white",
        linewidth=0.40,
        alpha=0.93,
        zorder=3,
    )

    ax.axhline(0, color="#333333", linewidth=0.75)
    ax.axhline(mean_diff, color=COLOR_FIT, linewidth=1.1, label="Mean bias")
    ax.axhline(
        loa_lower,
        color=COLOR_NEUTRAL,
        linestyle=(0, (4, 2)),
        linewidth=0.9,
        label="Approx. 95% limits of agreement",
    )
    ax.axhline(loa_upper, color=COLOR_NEUTRAL, linestyle=(0, (4, 2)), linewidth=0.9)

    ax.set_ylim(ba_ylim)
    ax.xaxis.set_major_formatter(FuncFormatter(axis_kha))
    ax.yaxis.set_major_formatter(FuncFormatter(axis_kha))
    ax.set_xlabel("Mean of mapped and official planted area (10³ ha)")
    ax.set_ylabel("Mapped − official planted area (10³ ha)")
    label_df = valid  # label all districts; adjustText below prevents collisions
    texts_ba = [
        ax.text(row["mean_area_ha"], row["difference_ha"], str(row[DISTRICT_EN_COL]),
                fontsize=6.2, color=COLOR_TEXT)
        for _, row in label_df.iterrows()
    ]
    adjust_text(
        texts_ba, ax=ax,
        x=label_df["mean_area_ha"].to_numpy(), y=label_df["difference_ha"].to_numpy(),
        arrowprops=dict(arrowstyle="-", color="#999999", lw=0.4),
        force_text=(0.8, 1.0), force_static=(0.4, 0.5), expand=(1.3, 1.6),
        max_move=(40, 40), time_lim=8,
    )
    style_axes(ax)
    add_panel_label(ax, panel_label, x=PANEL_LABEL_X_BA, y=PANEL_LABEL_Y_BA)


# =============================================================================
# 11. FIGURES
# =============================================================================

# Main Figure 9
fig = plt.figure(figsize=(7.2, 3.8))
gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.08], wspace=0.48)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])

draw_scatter(ax_a, panel_label="a")
draw_residuals(ax_b, panel_label="b")

plt.tight_layout()
for ext in SAVE_FORMATS:
    fig.savefig(
        OUT_DIR / f"Figure9_DistrictAreaConsistency.{ext}",
        dpi=DPI if ext == "png" else None,
        bbox_inches="tight",
    )
plt.close(fig)

# Standalone panels
standalone_specs = [
    ("Supplementary_Figure_S1_BlandAltman_DistrictAreaAgreement", draw_bland_altman, (6.0, 4.0)),
]

for filename, func, size in standalone_specs:
    fig, ax = plt.subplots(figsize=size)
    func(ax)
    plt.tight_layout()
    for ext in SAVE_FORMATS:
        fig.savefig(
            OUT_DIR / f"{filename}.{ext}",
            dpi=DPI if ext == "png" else None,
            bbox_inches="tight",
        )
    plt.close(fig)


# =============================================================================
# 12. CAPTIONS
# =============================================================================

caption = f"""Figure 9. District-level area consistency of mapped coffee extent in Dak Lak Province, Vietnam, in 2024.
(a) Comparison between mapped coffee area and official planted coffee area aggregated by district. The dashed line represents the 1:1 relationship, and the red line represents the fitted ordinary least-squares regression.
(b) District-level residuals, calculated as mapped area minus official planted coffee area. Negative values indicate underestimation, whereas positive values indicate overestimation. Across {len(valid)} districts, mapped and official areas showed strong area-level agreement (R² = {r2:.3f}; Pearson r = {pearson_r:.3f}), with RMSE = {fmt_ha(rmse_ha)} ha, MAE = {fmt_ha(mae_ha)} ha, MAPE = {mape_percent:.1f}% for districts with non-zero official area, and total bias = {signed_ha(bias_ha)} ha ({signed_pct(relative_bias_percent)}). The mapped coffee area totaled {fmt_ha(metrics_all["mapped_total_area_ha"])} ha, compared with {fmt_ha(metrics_all["official_total_area_ha"])} ha in official planted-area statistics, indicating a {bias_phrase}. This comparison is an external area-level consistency assessment and should not be interpreted as pixel-level classification accuracy.
"""

(SUPPLEMENTARY_DIR / "Figure9_Caption.txt").write_text(caption, encoding="utf-8")


# =============================================================================
# 13. CONSOLE REPORT
# =============================================================================

print("\n=== Figure 8 district-level area consistency complete ===")
print("\nInput files")
print(" Official:", official_path)
print(" GEE district:", gee_district_path)
print(" GEE class area:", gee_class_path)

print("\nArea audit")
print(area_audit.to_string(index=False))

print("\nCore metrics")
print(metrics.to_string(index=False))

print("\nBootstrap 95% CIs")
print(boot_summary.to_string(index=False))

print("\nOutputs saved in:", OUT_DIR.resolve())

print("\nMain Figure 8:")
print(" -", OUT_DIR / "Figure9_DistrictAreaConsistency.png")
print(" -", OUT_DIR / "Figure9_DistrictAreaConsistency.pdf")
print(" -", OUT_DIR / "Figure9_DistrictAreaConsistency.svg")

print("\nUpdated validation CSV:")
print(" -", SUPPLEMENTARY_DIR / "DistrictVsClassArea_Audit_20260715.csv")

print("\nFigure 9 caption:")
print(caption)


C:\Users\Owner\AppData\Local\Temp\ipykernel_5392\1125393904.py:904: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



=== Figure 8 district-level area consistency complete ===

Input files
 Official: data\raw\data_DakLak_Statistics\daklak_coffee_mapping_validation_2024.csv
 GEE district: data\raw\GEE_DakLak_Coffee_Area_By_District_2024.csv
 GEE class area: data\raw\Table_AreaStatistics_DakLak2024.csv

Area audit
                                                check  district_aggregation_ha  per_class_table_ha  difference_ha  relative_difference_percent
Total coffee: district aggregation vs per-class table            227720.740755       227720.740755  -2.910383e-11                -1.278049e-14
                                           Sun coffee            168126.658909       168126.658909   0.000000e+00                 0.000000e+00
                                     Intercrop coffee             49379.489080        49379.489080   0.000000e+00                 0.000000e+00
                                 Newly planted coffee             10214.592766        10214.592766   0.000000e+00                

## B. Supplementary SpatialBlockCV


In [4]:

# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path

# Main switch
RUN_SPATIAL_BLOCK_CV = True

# Keep audit strict but do not silently remove valid samples unless you choose to.
# Options: 'audit_only', 'drop_exact_duplicate_rows', 'drop_same_coordinate_same_class'
DEDUPE_MODE = 'audit_only'

# If True, rows outside the Dak Lak sanity bounding box are removed before modelling.
DROP_INVALID_GEOMETRY = True

# If True, drop features that are globally constant or entirely missing before CV.
# Fold-level near-zero and missing checks are still performed inside each fold.
DROP_GLOBAL_BAD_FEATURES = True

# Optional clipping of extreme values. Default False because the exported GEE features should remain unchanged.
CLIP_EXTREME_VALUES = False
CLIP_Q_LOW, CLIP_Q_HIGH = 0.001, 0.999

# Paths
INPUT_DIR = Path('data/raw')
OUTPUT_ROOT = SUPPLEMENTARY_DIR
AUDIT_DIR = SUPPLEMENTARY_DIR
CV_DIR = SUPPLEMENTARY_DIR
TABLE_DIR = TABLES_DIR
FIG_DIR = FIGURES_DIR
NOTE_DIR = SUPPLEMENTARY_DIR

# Required input files
TRAIN_FULL = 'Table_TrainSamples_FullFeatureSpace_2024.csv'
VAL_FULL = 'Table_ValSamples_FullFeatureSpace_2024.csv'

# Files that may exist but must not be pooled into SpatialBlockCV
IGNORED_FILES = [
    'Table_TrainSamples_RF_Final_2024.csv',
    'Table_ValSamples_RF_Final_2024.csv',
    'Table_ValPredictions_RF_Final_2024.csv',
    'Table_ValPredictions_RF_FullFeatures_2024.csv',
]

# Dak Lak sanity bounding box. This is only for detecting accidental coordinates outside the province.
DAKLAK_BBOX = {
    'lon_min': 107.20,
    'lon_max': 109.30,
    'lat_min': 11.95,
    'lat_max': 13.65,
}

# Feature prefixes used by the Paper 1 GEE export
FEATURE_PREFIXES = ('S1_', 'S2_', 'L89_', 'DEM_')
EXPECTED_N_FEATURES = 97
EXPECTED_N_SAMPLES = 3000
EXPECTED_N_CLASSES = 10
EXPECTED_N_PER_CLASS = 300

# Spatial CV parameters
BLOCK_SIZES_KM = [10, 15, 20]
N_SPLITS = 5
RANDOM_STATE = 2024

# Feature selection and model parameters
MAX_SELECTED_FEATURES = 25
CORR_THRESH = 0.90
MAX_TRAIN_MISSING_FRAC = 0.20
NEAR_ZERO_STD = 1e-12
RANK_TREES = 300
RF_TREES = 1500
RF_MAX_FEATURES = 3
RF_MAX_SAMPLES = 0.65
RF_MIN_SAMPLES_LEAF = 1
RF_CLASS_WEIGHT = 'balanced_subsample'

# Supplementary figure settings
PRIMARY_BLOCK_SIZE_KM = 15
TOP_N_FEATURES_FIG = 15
FIG_DPI = 300

# Class names used in the manuscript
CLASS_NAMES = {
    1: 'Sun coffee',
    2: 'Intercrop coffee',
    3: 'Newly planted coffee',
    4: 'Rubber',
    5: 'Partially vegetative',
    6: 'Rice',
    7: 'Other upland crops',
    8: 'Forest',
    9: 'Water',
    10: 'Built',
}
COFFEE_CLASSES = [1, 2, 3]
# Ensure output directories exist
for d in (OUTPUT_ROOT, AUDIT_DIR, CV_DIR, TABLE_DIR, FIG_DIR, NOTE_DIR):
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT


WindowsPath('D:/2024_PhD_Research/Chap2_Mapping/mmlab-coffeemap-daklak/results/supplementary')

In [5]:

# ============================================================
# 1. Imports and plotting defaults
# ============================================================
import json
import math
import shutil
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from pyproj import Transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.neighbors import BallTree, NearestNeighbors

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False

warnings.filterwarnings('ignore', category=RuntimeWarning)

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8.5,
    'figure.titlesize': 12,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'savefig.bbox': 'tight',
})



## 2. Input inventory and file contract

This section prevents a common mistake: accidentally combining FULL feature tables with reduced RF tables or validation-prediction tables. The SpatialBlockCV dataset must be constructed from the two FULL feature-space sample tables only.


In [6]:

# ============================================================
# 2. Input inventory
# ============================================================

def build_input_inventory(input_dir: Path) -> pd.DataFrame:
    rows = []
    for name in [TRAIN_FULL, VAL_FULL] + IGNORED_FILES:
        path = input_dir / name
        rows.append({
            'file': name,
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
            'role': 'USED_FOR_SPATIAL_BLOCK_CV' if name in [TRAIN_FULL, VAL_FULL] else 'IGNORED_TO_AVOID_DUPLICATION',
        })
    return pd.DataFrame(rows)

input_inventory = build_input_inventory(INPUT_DIR)
input_inventory.to_csv(AUDIT_DIR / 'S00_input_file_inventory.csv', index=False)
input_inventory


,file,exists,size_mb,role
0,Table_TrainSamples_FullFeatureSpace_2024.csv,True,2.132,USED_FOR_SPATIAL_BLOCK_CV
1,Table_ValSamples_FullFeatureSpace_2024.csv,True,0.914,USED_FOR_SPATIAL_BLOCK_CV
2,Table_TrainSamples_RF_Final_2024.csv,True,0.734,IGNORED_TO_AVOID_DUPLICATION
3,Table_ValSamples_RF_Final_2024.csv,True,0.315,IGNORED_TO_AVOID_DUPLICATION
4,Table_ValPredictions_RF_Final_2024.csv,True,0.316,IGNORED_TO_AVOID_DUPLICATION
5,Table_ValPredictions_RF_FullFeatures_2024.csv,True,0.916,IGNORED_TO_AVOID_DUPLICATION


In [7]:

# Hard stop if required files are missing
missing_required = input_inventory.query("role == 'USED_FOR_SPATIAL_BLOCK_CV' and exists == False")['file'].tolist()
if missing_required:
    raise FileNotFoundError(f'Missing required FULL feature-space files: {missing_required}')



## 3. Raw loading, schema standardisation, geometry parsing

The exported GEE `.geo` field is parsed into `lon` and `lat`. These coordinates are then checked against a broad Dak Lak bounding box and projected to UTM Zone 49N for metric block construction.


In [8]:

# ============================================================
# 3. Raw loading and geometry handling
# ============================================================

def parse_gee_point_geo(value) -> Tuple[float, float]:
    # Parse a GEE GeoJSON-like point stored as text and return lon/lat.
    if pd.isna(value):
        return (np.nan, np.nan)
    if isinstance(value, dict):
        obj = value
    else:
        try:
            obj = json.loads(str(value))
        except Exception:
            return (np.nan, np.nan)
    coords = obj.get('coordinates', None)
    if not isinstance(coords, (list, tuple)) or len(coords) < 2:
        return (np.nan, np.nan)
    try:
        return float(coords[0]), float(coords[1])
    except Exception:
        return (np.nan, np.nan)


def read_full_feature_tables(input_dir: Path) -> pd.DataFrame:
    train = pd.read_csv(input_dir / TRAIN_FULL, low_memory=False)
    val = pd.read_csv(input_dir / VAL_FULL, low_memory=False)
    train['split_origin'] = 'GEE_train_export'
    val['split_origin'] = 'GEE_validation_export'
    train['source_file'] = TRAIN_FULL
    val['source_file'] = VAL_FULL
    train['source_row'] = np.arange(len(train))
    val['source_row'] = np.arange(len(val))
    df = pd.concat([train, val], ignore_index=True)
    df['sample_uid'] = df['source_file'].str.replace('.csv', '', regex=False) + '_' + df['source_row'].astype(str)
    return df


def standardise_class_and_geometry(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'class_id' not in df.columns:
        raise ValueError('Required column class_id was not found.')
    df['class_id_raw'] = df['class_id']
    df['class_id'] = pd.to_numeric(df['class_id'], errors='coerce')

    if 'lon' not in df.columns or 'lat' not in df.columns:
        if '.geo' not in df.columns:
            raise ValueError('No lon/lat columns and no .geo column found.')
        coords = df['.geo'].apply(parse_gee_point_geo)
        df['lon'] = coords.apply(lambda z: z[0])
        df['lat'] = coords.apply(lambda z: z[1])
    else:
        df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
        df['lat'] = pd.to_numeric(df['lat'], errors='coerce')

    # Geometry validity flags
    df['valid_class'] = df['class_id'].between(1, EXPECTED_N_CLASSES) & df['class_id'].notna()
    df['valid_lonlat'] = df['lon'].between(-180, 180) & df['lat'].between(-90, 90)
    df['inside_daklak_bbox'] = (
        df['lon'].between(DAKLAK_BBOX['lon_min'], DAKLAK_BBOX['lon_max']) &
        df['lat'].between(DAKLAK_BBOX['lat_min'], DAKLAK_BBOX['lat_max'])
    )
    df['geometry_ok'] = df['valid_lonlat'] & df['inside_daklak_bbox']

    # Project valid coordinates to UTM 49N; leave invalid as NaN
    transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32649', always_xy=True)
    x = np.full(len(df), np.nan)
    y = np.full(len(df), np.nan)
    idx = df['valid_lonlat'].fillna(False).values
    if idx.sum() > 0:
        x_valid, y_valid = transformer.transform(df.loc[idx, 'lon'].values, df.loc[idx, 'lat'].values)
        x[idx] = x_valid
        y[idx] = y_valid
    df['x_utm'] = x
    df['y_utm'] = y
    df['class_id'] = df['class_id'].astype('Int64')
    return df

raw_df = read_full_feature_tables(INPUT_DIR)
df0 = standardise_class_and_geometry(raw_df)

schema_summary = pd.DataFrame({
    'item': ['raw_rows', 'raw_columns', 'valid_class_rows', 'valid_geometry_rows', 'inside_bbox_rows'],
    'value': [len(df0), df0.shape[1], int(df0['valid_class'].sum()), int(df0['valid_lonlat'].sum()), int(df0['inside_daklak_bbox'].sum())]
})
schema_summary.to_csv(AUDIT_DIR / 'S01_raw_schema_geometry_summary.csv', index=False)

invalid_rows = df0.loc[~(df0['valid_class'] & df0['geometry_ok'])].copy()
invalid_rows.to_csv(AUDIT_DIR / 'S01_invalid_class_or_geometry_rows.csv', index=False)

schema_summary


,item,value
0,raw_rows,3000
1,raw_columns,113
2,valid_class_rows,3000
3,valid_geometry_rows,3000
4,inside_bbox_rows,3000



## 4. Feature detection and type coercion

This section identifies the predictor columns exported from GEE. It also creates a feature dictionary by sensor/source and feature type, which becomes a supplementary table and a useful reproducibility check.


In [9]:

# ============================================================
# 4. Feature detection and feature dictionary
# ============================================================

def sensor_group(feature: str) -> str:
    if feature.startswith('S2_'):
        return 'Sentinel-2 optical'
    if feature.startswith('S1_'):
        return 'Sentinel-1 SAR'
    if feature.startswith('L89_'):
        return 'Landsat 8/9 optical'
    if feature.startswith('DEM_'):
        return 'DEM/topography'
    return 'Other'


def feature_season(feature: str) -> str:
    if '_dry_' in feature:
        return 'dry'
    if '_wet_' in feature:
        return 'wet'
    if 'asc' in feature.lower():
        return 'ascending'
    if 'desc' in feature.lower():
        return 'descending'
    return 'static_or_annual'


def feature_family(feature: str) -> str:
    f = feature.lower()
    if f.startswith('dem_'):
        return 'topographic'
    if 'glcm' in f:
        return 'texture'
    if any(k in f for k in ['ndvi', 'ndmi', 'ndwi', 'mndwi', 'nbr', 'nbr2', 'bsi', 'gndvi', 'cvi', 'mtci', 'ci_re', 'psri', 'rdvi', 'msr', 'sr', 'nli', 'ndi', 'msi']):
        return 'spectral_index'
    if any(k in f for k in ['vv', 'vh']):
        return 'sar_backscatter_metric'
    if any(k in f for k in ['blue', 'green', 'red', 'nir', 'swir', '_b2', '_b3', '_b4', '_b5', '_b6', '_b7', '_b8', '_b11', '_b12']):
        return 'spectral_band_or_percentile'
    return 'other'


def detect_and_coerce_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    df = df.copy()
    candidate_features = [c for c in df.columns if c.startswith(FEATURE_PREFIXES)]
    for c in candidate_features:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    feature_cols = [c for c in candidate_features if pd.api.types.is_numeric_dtype(df[c])]
    feature_cols = sorted(feature_cols)

    dictionary = pd.DataFrame({
        'feature': feature_cols,
    })
    dictionary['source_group'] = dictionary['feature'].map(sensor_group)
    dictionary['season_or_orbit'] = dictionary['feature'].map(feature_season)
    dictionary['feature_family'] = dictionary['feature'].map(feature_family)
    dictionary = dictionary.sort_values(['source_group', 'season_or_orbit', 'feature_family', 'feature']).reset_index(drop=True)
    return df, feature_cols, dictionary

df1, feature_cols_raw, feature_dictionary = detect_and_coerce_features(df0)
feature_dictionary.to_csv(AUDIT_DIR / 'S02_feature_dictionary_FULL97.csv', index=False)

feature_group_counts = (
    feature_dictionary.groupby(['source_group', 'season_or_orbit', 'feature_family'])
    .size().reset_index(name='n_features')
    .sort_values(['source_group', 'season_or_orbit', 'feature_family'])
)
feature_group_counts.to_csv(AUDIT_DIR / 'S02_feature_group_counts.csv', index=False)

print(f'Detected {len(feature_cols_raw)} feature columns')
feature_group_counts


Detected 97 feature columns


,source_group,season_or_orbit,feature_family,n_features
0,DEM/topography,static_or_annual,topographic,4
1,Landsat 8/9 optical,dry,spectral_band_or_percentile,7
2,Landsat 8/9 optical,dry,spectral_index,6
3,Landsat 8/9 optical,wet,spectral_band_or_percentile,7
4,Landsat 8/9 optical,wet,spectral_index,6
5,Sentinel-1 SAR,ascending,sar_backscatter_metric,3
6,Sentinel-1 SAR,descending,sar_backscatter_metric,9
7,Sentinel-1 SAR,static_or_annual,texture,3
8,Sentinel-2 optical,dry,spectral_band_or_percentile,9
9,Sentinel-2 optical,dry,spectral_index,17


In [10]:

# Optional hard warning when the expected FULL97 stack is not present.
if len(feature_cols_raw) != EXPECTED_N_FEATURES:
    print(f'WARNING: Expected {EXPECTED_N_FEATURES} predictors but detected {len(feature_cols_raw)}.')
else:
    print('Feature-count check passed: FULL97 detected.')


Feature-count check passed: FULL97 detected.



## 5. Sample-level data QA and deduplication audit

This block checks whether the pooled 3,000 points are balanced, whether any labels are invalid, and whether duplicate or near-duplicate samples exist. By default, the notebook audits duplicates but does not remove them unless `DEDUPE_MODE` is changed.


In [11]:

# ============================================================
# 5. Sample-level QA and duplicate audit
# ============================================================

def sample_qa_tables(df: pd.DataFrame, feature_cols: List[str]) -> Dict[str, pd.DataFrame]:
    tables = {}
    tables['class_counts_total'] = (
        df.groupby('class_id', dropna=False).size().reset_index(name='n')
        .assign(class_name=lambda z: z['class_id'].map(CLASS_NAMES))
    )
    tables['class_counts_by_origin'] = (
        df.groupby(['split_origin', 'class_id'], dropna=False).size().reset_index(name='n')
        .assign(class_name=lambda z: z['class_id'].map(CLASS_NAMES))
    )
    tables['origin_counts'] = df.groupby('split_origin').size().reset_index(name='n')
    return tables


def duplicate_audit(df: pd.DataFrame, feature_cols: List[str]) -> Dict[str, pd.DataFrame]:
    df = df.copy()
    # Exact row duplicate over class + feature + approximate geometry
    exact_subset = ['class_id', 'lon', 'lat'] + feature_cols
    exact_subset = [c for c in exact_subset if c in df.columns]
    exact_mask = df.duplicated(subset=exact_subset, keep=False)
    exact_dups = df.loc[exact_mask, ['sample_uid', 'source_file', 'source_row', 'split_origin', 'class_id', 'lon', 'lat']].copy()

    # Coordinate duplicate at roughly sub-meter scale
    df['coord_key_6dp'] = df['lon'].round(6).astype(str) + '_' + df['lat'].round(6).astype(str)
    coord_counts = (
        df.groupby('coord_key_6dp')
        .agg(n=('sample_uid', 'size'), n_classes=('class_id', 'nunique'), classes=('class_id', lambda s: ','.join(map(str, sorted(pd.Series(s).dropna().unique())))))
        .reset_index()
        .query('n > 1')
        .sort_values(['n_classes', 'n'], ascending=[False, False])
    )
    coord_dups = df[df['coord_key_6dp'].isin(coord_counts['coord_key_6dp'])][
        ['sample_uid', 'source_file', 'source_row', 'split_origin', 'class_id', 'lon', 'lat', 'coord_key_6dp']
    ].copy()

    # Near-duplicate audit within 10 m using UTM coordinates
    valid = df.dropna(subset=['x_utm', 'y_utm']).copy()
    near_pairs = []
    if len(valid) > 1:
        coords = valid[['x_utm', 'y_utm']].values
        tree = BallTree(coords, metric='euclidean')
        ind = tree.query_radius(coords, r=10.0)
        for i, neigh in enumerate(ind):
            for j in neigh:
                if j <= i:
                    continue
                near_pairs.append({
                    'sample_uid_1': valid.iloc[i]['sample_uid'],
                    'sample_uid_2': valid.iloc[j]['sample_uid'],
                    'class_id_1': valid.iloc[i]['class_id'],
                    'class_id_2': valid.iloc[j]['class_id'],
                    'distance_m': float(np.linalg.norm(coords[i] - coords[j])),
                    'label_conflict': valid.iloc[i]['class_id'] != valid.iloc[j]['class_id'],
                })
    near_pairs = pd.DataFrame(near_pairs)
    return {'exact_duplicates': exact_dups, 'coordinate_duplicates': coord_dups, 'coordinate_duplicate_summary': coord_counts, 'near_duplicate_pairs_10m': near_pairs}

# Start with valid geometry/class rows only if requested
if DROP_INVALID_GEOMETRY:
    df_model_base = df1.loc[df1['valid_class'] & df1['geometry_ok']].copy()
else:
    df_model_base = df1.loc[df1['valid_class'] & df1['valid_lonlat']].copy()

df_model_base['class_id'] = df_model_base['class_id'].astype(int)

qa_tables = sample_qa_tables(df_model_base, feature_cols_raw)
for name, table in qa_tables.items():
    table.to_csv(AUDIT_DIR / f'S03_{name}.csv', index=False)

dup_tables = duplicate_audit(df_model_base, feature_cols_raw)
for name, table in dup_tables.items():
    table.to_csv(AUDIT_DIR / f'S03_{name}.csv', index=False)

print('Rows after class/geometry filter:', len(df_model_base))
print('Class counts:')
print(qa_tables['class_counts_total'].to_string(index=False))
print('\nDuplicate audit:')
print({k: len(v) for k, v in dup_tables.items()})


Rows after class/geometry filter: 3000
Class counts:
 class_id   n           class_name
        1 300           Sun coffee
        2 300     Intercrop coffee
        3 300 Newly planted coffee
        4 300               Rubber
        5 300 Partially vegetative
        6 300                 Rice
        7 300   Other upland crops
        8 300               Forest
        9 300                Water
       10 300                Built

Duplicate audit:
{'exact_duplicates': 256, 'coordinate_duplicates': 256, 'coordinate_duplicate_summary': 125, 'near_duplicate_pairs_10m': 928}


In [12]:

# Apply optional deduplication mode

def apply_deduplication(df: pd.DataFrame, feature_cols: List[str], mode: str) -> pd.DataFrame:
    df = df.copy()
    if mode == 'audit_only':
        return df
    if mode == 'drop_exact_duplicate_rows':
        subset = ['class_id', 'lon', 'lat'] + feature_cols
        subset = [c for c in subset if c in df.columns]
        return df.drop_duplicates(subset=subset, keep='first').copy()
    if mode == 'drop_same_coordinate_same_class':
        df['coord_key_6dp'] = df['lon'].round(6).astype(str) + '_' + df['lat'].round(6).astype(str)
        return df.drop_duplicates(subset=['coord_key_6dp', 'class_id'], keep='first').drop(columns=['coord_key_6dp']).copy()
    raise ValueError(f'Unknown DEDUPE_MODE: {mode}')

df2 = apply_deduplication(df_model_base, feature_cols_raw, DEDUPE_MODE)
print(f'Rows after deduplication mode = {DEDUPE_MODE}: {len(df2):,}')


Rows after deduplication mode = audit_only: 3,000



## 6. Feature-level QA, missing values, zero proportions and range audit

This section exports the predictor audit table used to justify that the model used the intended FULL97 feature stack. It also identifies features that would be unsafe for modelling if they were entirely missing or constant.


In [13]:

# ============================================================
# 6. Feature-level QA
# ============================================================

def feature_audit(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    rows = []
    n = len(df)
    for c in feature_cols:
        s = pd.to_numeric(df[c], errors='coerce')
        finite = np.isfinite(s)
        s_finite = s[finite]
        rows.append({
            'feature': c,
            'source_group': sensor_group(c),
            'season_or_orbit': feature_season(c),
            'feature_family': feature_family(c),
            'n': n,
            'missing_count': int(s.isna().sum()),
            'missing_pct': float(100 * s.isna().sum() / n),
            'inf_count': int(np.isinf(s).sum()),
            'zero_count': int((s == 0).sum(skipna=True)),
            'zero_pct': float(100 * (s == 0).sum(skipna=True) / n),
            'finite_count': int(finite.sum()),
            'mean': float(s_finite.mean()) if len(s_finite) else np.nan,
            'std': float(s_finite.std()) if len(s_finite) else np.nan,
            'min': float(s_finite.min()) if len(s_finite) else np.nan,
            'p01': float(s_finite.quantile(0.01)) if len(s_finite) else np.nan,
            'p25': float(s_finite.quantile(0.25)) if len(s_finite) else np.nan,
            'median': float(s_finite.quantile(0.50)) if len(s_finite) else np.nan,
            'p75': float(s_finite.quantile(0.75)) if len(s_finite) else np.nan,
            'p99': float(s_finite.quantile(0.99)) if len(s_finite) else np.nan,
            'max': float(s_finite.max()) if len(s_finite) else np.nan,
            'all_missing': bool(s.isna().all()),
            'constant_or_near_constant': bool((len(s_finite) == 0) or (s_finite.std() <= NEAR_ZERO_STD)),
        })
    return pd.DataFrame(rows)

feat_audit = feature_audit(df2, feature_cols_raw)
feat_audit.to_csv(AUDIT_DIR / 'S04_feature_missing_zero_range_audit.csv', index=False)

bad_global_features = feat_audit.query('all_missing == True or constant_or_near_constant == True')['feature'].tolist()
if DROP_GLOBAL_BAD_FEATURES:
    feature_cols = [c for c in feature_cols_raw if c not in bad_global_features]
else:
    feature_cols = feature_cols_raw.copy()

if CLIP_EXTREME_VALUES:
    df2 = df2.copy()
    for c in feature_cols:
        lo = df2[c].quantile(CLIP_Q_LOW)
        hi = df2[c].quantile(CLIP_Q_HIGH)
        df2[c] = df2[c].clip(lo, hi)

print(f'Raw features: {len(feature_cols_raw)}')
print(f'Global bad features: {len(bad_global_features)}')
print(f'Features used for CV: {len(feature_cols)}')
feat_audit.sort_values(['missing_pct', 'constant_or_near_constant'], ascending=[False, False]).head(10)


Raw features: 97


Global bad features: 0
Features used for CV: 97


,feature,source_group,season_or_orbit,feature_family,n,missing_count,missing_pct,inf_count,zero_count,zero_pct,...,std,min,p01,p25,median,p75,p99,max,all_missing,constant_or_near_constant
0,DEM_aspect,DEM/topography,static_or_annual,topographic,3000,0,0.0,0,275,9.166667,...,106.215974,0.000000,0.000000,90.000000,180.000000,263.000000,346.000000,358.000000,False,False
1,DEM_elevation,DEM/topography,static_or_annual,topographic,3000,0,0.0,0,0,0.000000,...,181.728258,147.000000,157.000000,443.000000,531.000000,584.000000,1163.020000,1824.000000,False,False
2,DEM_hillshade,DEM/topography,static_or_annual,topographic,3000,0,0.0,0,0,0.000000,...,15.530380,84.000000,132.990000,177.000000,180.000000,189.000000,245.010000,255.000000,False,False
3,DEM_slope,DEM/topography,static_or_annual,topographic,3000,0,0.0,0,115,3.833333,...,5.363268,0.000000,0.000000,2.000000,3.000000,5.000000,28.000000,51.000000,False,False
4,L89_dry_BSI,Landsat 8/9 optical,dry,spectral_index,3000,0,0.0,0,57,1.900000,...,0.171950,-0.446282,-0.404703,-0.241825,-0.097541,0.067605,0.228065,0.419159,False,False
5,L89_dry_Blue,Landsat 8/9 optical,dry,spectral_band_or_percentile,3000,0,0.0,0,57,1.900000,...,0.018472,0.000000,0.000000,0.023658,0.027404,0.037552,0.090872,0.121173,False,False
6,L89_dry_Green,Landsat 8/9 optical,dry,spectral_band_or_percentile,3000,0,0.0,0,57,1.900000,...,0.025633,0.000000,0.000000,0.047510,0.053192,0.067245,0.141310,0.180875,False,False
7,L89_dry_MSI,Landsat 8/9 optical,dry,spectral_index,3000,0,0.0,0,57,1.900000,...,0.305956,0.000000,0.000000,0.506453,0.662146,0.891385,1.471177,3.129733,False,False
8,L89_dry_NBR,Landsat 8/9 optical,dry,spectral_index,3000,0,0.0,0,57,1.900000,...,0.243254,-0.217538,-0.120441,0.185688,0.448756,0.605392,0.749505,0.904001,False,False
9,L89_dry_NBR2,Landsat 8/9 optical,dry,spectral_index,3000,0,0.0,0,57,1.900000,...,0.115052,-0.026307,0.000000,0.149182,0.278188,0.344220,0.436887,0.459998,False,False


In [14]:

# Final pre-CV integrity report
precv_report = {
    'n_samples_model': len(df2),
    'n_features_raw': len(feature_cols_raw),
    'n_features_used': len(feature_cols),
    'dedupe_mode': DEDUPE_MODE,
    'drop_invalid_geometry': DROP_INVALID_GEOMETRY,
    'drop_global_bad_features': DROP_GLOBAL_BAD_FEATURES,
    'n_bad_global_features': len(bad_global_features),
    'class_counts': df2['class_id'].value_counts().sort_index().to_dict(),
}

with open(AUDIT_DIR / 'S05_pre_cv_integrity_report.json', 'w', encoding='utf-8') as f:
    json.dump(precv_report, f, indent=2, ensure_ascii=False)

# Soft checks: these warn rather than hard-stop because deduplication settings may intentionally alter counts.
if len(df2) != EXPECTED_N_SAMPLES:
    print(f'WARNING: expected {EXPECTED_N_SAMPLES:,} samples, found {len(df2):,}.')
class_counts = df2['class_id'].value_counts().sort_index()
if len(class_counts) != EXPECTED_N_CLASSES or not (class_counts == EXPECTED_N_PER_CLASS).all():
    print('WARNING: class balance differs from 300 samples per class:')
    print(class_counts)
else:
    print('Sample-balance check passed: 300 samples per class.')


Sample-balance check passed: 300 samples per class.



## 7. Spatial block construction and fold diagnostics

Blocks are generated in UTM metres, not degrees. The notebook exports block summaries for each block size and records fold-level leakage diagnostics using nearest train-test sample distances.


In [15]:

# ============================================================
# 7. Spatial blocks and diagnostics
# ============================================================

def assign_spatial_blocks(df: pd.DataFrame, block_size_km: int, x_offset_m: float = 0, y_offset_m: float = 0) -> pd.DataFrame:
    df = df.copy()
    block_m = int(block_size_km * 1000)
    df['block_size_km'] = block_size_km
    df['block_x'] = np.floor((df['x_utm'] - x_offset_m) / block_m).astype(int)
    df['block_y'] = np.floor((df['y_utm'] - y_offset_m) / block_m).astype(int)
    df['block_id'] = df['block_x'].astype(str) + '_' + df['block_y'].astype(str)
    return df


def block_summary_tables(df_blocked: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    block_summary = (
        df_blocked.groupby('block_id')
        .agg(
            n_samples=('sample_uid', 'size'),
            n_classes=('class_id', 'nunique'),
            lon_mean=('lon', 'mean'),
            lat_mean=('lat', 'mean'),
            x_mean=('x_utm', 'mean'),
            y_mean=('y_utm', 'mean'),
        )
        .reset_index()
        .sort_values(['n_samples', 'n_classes'], ascending=[False, False])
    )
    block_class = (
        df_blocked.groupby(['block_id', 'class_id'])
        .size().reset_index(name='n_samples')
        .assign(class_name=lambda z: z['class_id'].map(CLASS_NAMES))
        .sort_values(['block_id', 'class_id'])
    )
    return {'block_summary': block_summary, 'block_class_counts': block_class}

for km in BLOCK_SIZES_KM:
    tmp = assign_spatial_blocks(df2, km)
    tabs = block_summary_tables(tmp)
    tabs['block_summary'].to_csv(AUDIT_DIR / f'S06_block_summary_{km}km.csv', index=False)
    tabs['block_class_counts'].to_csv(AUDIT_DIR / f'S06_block_class_counts_{km}km.csv', index=False)
    print(f'{km} km: {tmp["block_id"].nunique()} blocks; median samples/block = {tabs["block_summary"]["n_samples"].median():.1f}')


10 km: 102 blocks; median samples/block = 8.0
15 km: 59 blocks; median samples/block = 15.0
20 km: 39 blocks; median samples/block = 23.0



## 8. Fold-safe preprocessing, feature selection and Random Forest modelling

Key design rule: **no information from the held-out spatial fold is used for imputation, feature ranking, correlation filtering, or classifier fitting**. This is stricter than a simple random train/test split and is more appropriate for assessing spatial generalisation.


In [16]:

# ============================================================
# 8. Model helper functions
# ============================================================

def fold_candidate_features(X_train_raw: pd.DataFrame, feature_cols: List[str]) -> List[str]:
    keep = []
    for c in feature_cols:
        s = pd.to_numeric(X_train_raw[c], errors='coerce')
        missing_frac = s.isna().mean()
        finite_s = s[np.isfinite(s)]
        if missing_frac > MAX_TRAIN_MISSING_FRAC:
            continue
        if len(finite_s) == 0:
            continue
        if finite_s.std() <= NEAR_ZERO_STD:
            continue
        keep.append(c)
    return keep


def fit_fold_imputer(X_train_raw: pd.DataFrame, X_test_raw: pd.DataFrame, candidate_features: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame, SimpleImputer]:
    imputer = SimpleImputer(strategy='median')
    Xtr = pd.DataFrame(
        imputer.fit_transform(X_train_raw[candidate_features]),
        columns=candidate_features,
        index=X_train_raw.index,
    )
    Xte = pd.DataFrame(
        imputer.transform(X_test_raw[candidate_features]),
        columns=candidate_features,
        index=X_test_raw.index,
    )
    return Xtr, Xte, imputer


def rank_features_rf(X_train: pd.DataFrame, y_train: np.ndarray, candidate_features: List[str], seed: int) -> pd.DataFrame:
    rf = RandomForestClassifier(
        n_estimators=RANK_TREES,
        max_features=min(RF_MAX_FEATURES, len(candidate_features)),
        bootstrap=True,
        max_samples=RF_MAX_SAMPLES,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        class_weight=RF_CLASS_WEIGHT,
        random_state=seed,
        n_jobs=-1,
    )
    rf.fit(X_train[candidate_features], y_train)
    ranking = pd.DataFrame({
        'feature': candidate_features,
        'ranking_importance': rf.feature_importances_,
    }).sort_values('ranking_importance', ascending=False).reset_index(drop=True)
    ranking['rank'] = np.arange(1, len(ranking) + 1)
    return ranking


def greedy_correlation_filter(X_train: pd.DataFrame, ranked_features: List[str], corr_thresh: float, max_features: int) -> List[str]:
    selected = []
    if len(ranked_features) == 0:
        return selected
    corr = X_train[ranked_features].corr(method='pearson').abs()
    for feat in ranked_features:
        if len(selected) >= max_features:
            break
        if not selected:
            selected.append(feat)
            continue
        max_corr = corr.loc[feat, selected].max()
        if pd.isna(max_corr) or max_corr < corr_thresh:
            selected.append(feat)
    return selected


def classwise_metrics(y_true: np.ndarray, y_pred: np.ndarray, labels: Sequence[int]) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rowsum = cm.sum(axis=1)
    colsum = cm.sum(axis=0)
    diag = np.diag(cm)
    rows = []
    for i, lab in enumerate(labels):
        pa = diag[i] / rowsum[i] if rowsum[i] > 0 else np.nan
        ua = diag[i] / colsum[i] if colsum[i] > 0 else np.nan
        f1 = 2 * pa * ua / (pa + ua) if (pa + ua) > 0 else np.nan
        rows.append({
            'class_id': lab,
            'class_name': CLASS_NAMES.get(lab, str(lab)),
            'producer_accuracy_recall': pa,
            'user_accuracy_precision': ua,
            'f1': f1,
            'support_true': int(rowsum[i]),
            'support_pred': int(colsum[i]),
        })
    return pd.DataFrame(rows)


def fold_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    true_bin = np.isin(y_true, COFFEE_CLASSES).astype(int)
    pred_bin = np.isin(y_pred, COFFEE_CLASSES).astype(int)
    return {
        'OA': accuracy_score(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred),
        'MacroF1_10class': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'CoffeeSubclassMacroF1': f1_score(y_true, y_pred, labels=COFFEE_CLASSES, average='macro', zero_division=0),
        'CoffeeBinaryF1': f1_score(true_bin, pred_bin, zero_division=0),
        'CoffeeBinaryPrecision': precision_score(true_bin, pred_bin, zero_division=0),
        'CoffeeBinaryRecall': recall_score(true_bin, pred_bin, zero_division=0),
        'CoffeeBinaryOA': accuracy_score(true_bin, pred_bin),
    }


def train_test_distance_diagnostics(df_blocked: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray) -> Dict[str, float]:
    train_xy = df_blocked.iloc[train_idx][['x_utm', 'y_utm']].values
    test_xy = df_blocked.iloc[test_idx][['x_utm', 'y_utm']].values
    nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn.fit(train_xy)
    dist, _ = nn.kneighbors(test_xy)
    dist = dist.ravel()
    return {
        'nearest_train_test_distance_min_m': float(np.min(dist)),
        'nearest_train_test_distance_p05_m': float(np.quantile(dist, 0.05)),
        'nearest_train_test_distance_median_m': float(np.median(dist)),
        'nearest_train_test_distance_mean_m': float(np.mean(dist)),
    }


def make_splitter():
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE), 'StratifiedGroupKFold'
    return GroupKFold(n_splits=N_SPLITS), 'GroupKFold'


In [17]:

# ============================================================
# 9. Run SpatialBlockCV
# ============================================================

def run_spatial_block_cv(df: pd.DataFrame, feature_cols: List[str]) -> Dict[str, pd.DataFrame]:
    all_fold_metrics = []
    all_class_metrics = []
    all_selected = []
    all_rankings = []
    all_final_importances = []
    all_predictions = []
    all_fold_class_counts = []
    all_confusions = []

    labels = list(range(1, EXPECTED_N_CLASSES + 1))

    for block_size_km in BLOCK_SIZES_KM:
        dfb = assign_spatial_blocks(df, block_size_km)
        splitter, splitter_name = make_splitter()

        X_all = dfb[feature_cols].copy()
        y_all = dfb['class_id'].astype(int).values
        groups = dfb['block_id'].values

        if dfb['block_id'].nunique() < N_SPLITS:
            raise ValueError(f'{block_size_km} km has fewer unique blocks than N_SPLITS.')

        for fold, (train_idx, test_idx) in enumerate(splitter.split(X_all, y_all, groups=groups), start=1):
            seed = RANDOM_STATE + block_size_km * 100 + fold
            X_train_raw = X_all.iloc[train_idx].copy()
            X_test_raw = X_all.iloc[test_idx].copy()
            y_train = y_all[train_idx]
            y_test = y_all[test_idx]

            candidate = fold_candidate_features(X_train_raw, feature_cols)
            if len(candidate) == 0:
                raise ValueError(f'No usable candidate predictors in block {block_size_km} km fold {fold}.')
            X_train, X_test, imputer = fit_fold_imputer(X_train_raw, X_test_raw, candidate)

            ranking = rank_features_rf(X_train, y_train, candidate, seed=seed)
            ranked_features = ranking['feature'].tolist()
            selected = greedy_correlation_filter(
                X_train,
                ranked_features,
                corr_thresh=CORR_THRESH,
                max_features=min(MAX_SELECTED_FEATURES, len(ranked_features)),
            )

            rf = RandomForestClassifier(
                n_estimators=RF_TREES,
                max_features=min(RF_MAX_FEATURES, len(selected)),
                bootstrap=True,
                max_samples=RF_MAX_SAMPLES,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                class_weight=RF_CLASS_WEIGHT,
                oob_score=True,
                random_state=seed,
                n_jobs=-1,
            )
            rf.fit(X_train[selected], y_train)
            y_pred = rf.predict(X_test[selected])
            proba = rf.predict_proba(X_test[selected])

            # Fold metrics
            m = fold_metrics(y_test, y_pred)
            m.update({
                'block_size_km': block_size_km,
                'fold': fold,
                'splitter': splitter_name,
                'n_train_samples': int(len(train_idx)),
                'n_test_samples': int(len(test_idx)),
                'n_train_blocks': int(len(np.unique(groups[train_idx]))),
                'n_test_blocks': int(len(np.unique(groups[test_idx]))),
                'n_candidate_features_after_fold_screen': int(len(candidate)),
                'n_selected_features': int(len(selected)),
                'oob_score': float(rf.oob_score_) if hasattr(rf, 'oob_score_') else np.nan,
            })
            m.update(train_test_distance_diagnostics(dfb, train_idx, test_idx))
            all_fold_metrics.append(m)

            # Class metrics by fold
            cls = classwise_metrics(y_test, y_pred, labels=labels)
            cls.insert(0, 'fold', fold)
            cls.insert(0, 'block_size_km', block_size_km)
            all_class_metrics.append(cls)

            # Fold class distribution
            fc = pd.DataFrame({'class_id': labels})
            fc['class_name'] = fc['class_id'].map(CLASS_NAMES)
            train_counts = pd.Series(y_train).value_counts().reindex(labels, fill_value=0).values
            test_counts = pd.Series(y_test).value_counts().reindex(labels, fill_value=0).values
            fc['n_train'] = train_counts
            fc['n_test'] = test_counts
            fc.insert(0, 'fold', fold)
            fc.insert(0, 'block_size_km', block_size_km)
            all_fold_class_counts.append(fc)

            # Selected features and rankings
            sf = pd.DataFrame({
                'block_size_km': block_size_km,
                'fold': fold,
                'rank_selected': np.arange(1, len(selected) + 1),
                'feature': selected,
            })
            sf['source_group'] = sf['feature'].map(sensor_group)
            sf['season_or_orbit'] = sf['feature'].map(feature_season)
            sf['feature_family'] = sf['feature'].map(feature_family)
            all_selected.append(sf)

            ranking_out = ranking.copy()
            ranking_out.insert(0, 'fold', fold)
            ranking_out.insert(0, 'block_size_km', block_size_km)
            ranking_out['selected'] = ranking_out['feature'].isin(selected)
            ranking_out['source_group'] = ranking_out['feature'].map(sensor_group)
            ranking_out['season_or_orbit'] = ranking_out['feature'].map(feature_season)
            ranking_out['feature_family'] = ranking_out['feature'].map(feature_family)
            all_rankings.append(ranking_out)

            final_imp = pd.DataFrame({
                'block_size_km': block_size_km,
                'fold': fold,
                'feature': selected,
                'final_rf_importance': rf.feature_importances_,
            }).sort_values('final_rf_importance', ascending=False)
            final_imp['source_group'] = final_imp['feature'].map(sensor_group)
            final_imp['season_or_orbit'] = final_imp['feature'].map(feature_season)
            final_imp['feature_family'] = final_imp['feature'].map(feature_family)
            all_final_importances.append(final_imp)

            # Predictions with probability columns
            pred_df = dfb.iloc[test_idx][['sample_uid', 'source_file', 'source_row', 'split_origin', 'class_id', 'lon', 'lat', 'x_utm', 'y_utm', 'block_id']].copy()
            pred_df = pred_df.rename(columns={'class_id': 'y_true'})
            pred_df['block_size_km'] = block_size_km
            pred_df['fold'] = fold
            pred_df['y_pred'] = y_pred
            for class_id in labels:
                pred_df[f'prob_class_{class_id}'] = 0.0
            for j, class_id in enumerate(rf.classes_):
                pred_df[f'prob_class_{int(class_id)}'] = proba[:, j]
            pred_df['coffee_true'] = pred_df['y_true'].isin(COFFEE_CLASSES).astype(int)
            pred_df['coffee_pred'] = pred_df['y_pred'].isin(COFFEE_CLASSES).astype(int)
            all_predictions.append(pred_df)

            cm = confusion_matrix(y_test, y_pred, labels=labels)
            cm_df = pd.DataFrame(cm, index=[f'true_{i}' for i in labels], columns=[f'pred_{i}' for i in labels])
            cm_df.insert(0, 'fold', fold)
            cm_df.insert(0, 'block_size_km', block_size_km)
            all_confusions.append(cm_df.reset_index(names='true_class'))

            print(f'Block {block_size_km:>2} km | fold {fold}: OA={m["OA"]:.3f}, MacroF1={m["MacroF1_10class"]:.3f}, selected={len(selected)}')

    fold_metrics_df = pd.DataFrame(all_fold_metrics).sort_values(['block_size_km', 'fold'])
    class_metrics_df = pd.concat(all_class_metrics, ignore_index=True)
    selected_df = pd.concat(all_selected, ignore_index=True)
    rankings_df = pd.concat(all_rankings, ignore_index=True)
    final_imp_df = pd.concat(all_final_importances, ignore_index=True)
    predictions_df = pd.concat(all_predictions, ignore_index=True)
    fold_class_counts_df = pd.concat(all_fold_class_counts, ignore_index=True)
    confusions_df = pd.concat(all_confusions, ignore_index=True)

    summary_df = (
        fold_metrics_df.groupby('block_size_km', as_index=False)
        .agg(
            n_folds=('fold', 'count'),
            n_selected_features_mean=('n_selected_features', 'mean'),
            OA_mean=('OA', 'mean'), OA_sd=('OA', 'std'),
            Kappa_mean=('Kappa', 'mean'), Kappa_sd=('Kappa', 'std'),
            MacroF1_10class_mean=('MacroF1_10class', 'mean'), MacroF1_10class_sd=('MacroF1_10class', 'std'),
            CoffeeSubclassMacroF1_mean=('CoffeeSubclassMacroF1', 'mean'), CoffeeSubclassMacroF1_sd=('CoffeeSubclassMacroF1', 'std'),
            CoffeeBinaryF1_mean=('CoffeeBinaryF1', 'mean'), CoffeeBinaryF1_sd=('CoffeeBinaryF1', 'std'),
            CoffeeBinaryPrecision_mean=('CoffeeBinaryPrecision', 'mean'), CoffeeBinaryPrecision_sd=('CoffeeBinaryPrecision', 'std'),
            CoffeeBinaryRecall_mean=('CoffeeBinaryRecall', 'mean'), CoffeeBinaryRecall_sd=('CoffeeBinaryRecall', 'std'),
            nearest_train_test_distance_median_m_mean=('nearest_train_test_distance_median_m', 'mean'),
        )
        .sort_values('block_size_km')
    )

    class_summary_df = (
        class_metrics_df.groupby(['block_size_km', 'class_id', 'class_name'], as_index=False)
        .agg(
            producer_accuracy_mean=('producer_accuracy_recall', 'mean'),
            producer_accuracy_sd=('producer_accuracy_recall', 'std'),
            user_accuracy_mean=('user_accuracy_precision', 'mean'),
            user_accuracy_sd=('user_accuracy_precision', 'std'),
            f1_mean=('f1', 'mean'),
            f1_sd=('f1', 'std'),
            support_true_mean=('support_true', 'mean'),
            support_pred_mean=('support_pred', 'mean'),
        )
        .sort_values(['block_size_km', 'class_id'])
    )

    selection_frequency_df = (
        selected_df.groupby(['block_size_km', 'feature', 'source_group', 'season_or_orbit', 'feature_family'], as_index=False)
        .size()
        .rename(columns={'size': 'selected_count_out_of_folds'})
        .sort_values(['block_size_km', 'selected_count_out_of_folds', 'feature'], ascending=[True, False, True])
    )
    selection_frequency_df['selection_frequency_pct'] = 100 * selection_frequency_df['selected_count_out_of_folds'] / N_SPLITS

    outputs = {
        'fold_metrics': fold_metrics_df,
        'summary': summary_df,
        'class_metrics_by_fold': class_metrics_df,
        'class_metrics_summary': class_summary_df,
        'selected_features_by_fold': selected_df,
        'feature_rankings_by_fold': rankings_df,
        'final_rf_importance_by_fold': final_imp_df,
        'selection_frequency': selection_frequency_df,
        'predictions': predictions_df,
        'fold_class_counts': fold_class_counts_df,
        'confusion_matrices_by_fold': confusions_df,
    }
    return outputs

if RUN_SPATIAL_BLOCK_CV:
    cv_outputs = run_spatial_block_cv(df2, feature_cols)
    for name, table in cv_outputs.items():
        table.to_csv(CV_DIR / f'{name}.csv', index=False)
else:
    cv_outputs = {}
    for name in ['fold_metrics', 'summary', 'class_metrics_by_fold', 'class_metrics_summary', 'selected_features_by_fold', 'feature_rankings_by_fold', 'final_rf_importance_by_fold', 'selection_frequency', 'predictions', 'fold_class_counts', 'confusion_matrices_by_fold']:
        cv_outputs[name] = pd.read_csv(CV_DIR / f'{name}.csv')

cv_outputs['summary']


Block 10 km | fold 1: OA=0.650, MacroF1=0.630, selected=25


Block 10 km | fold 2: OA=0.624, MacroF1=0.651, selected=25


Block 10 km | fold 3: OA=0.872, MacroF1=0.781, selected=25


Block 10 km | fold 4: OA=0.821, MacroF1=0.783, selected=25


Block 10 km | fold 5: OA=0.842, MacroF1=0.749, selected=25


...

Block 20 km | fold 2: OA=0.858, MacroF1=0.644, selected=25


Block 20 km | fold 3: OA=0.537, MacroF1=0.604, selected=25


Block 20 km | fold 4: OA=0.954, MacroF1=0.647, selected=25


Block 20 km | fold 5: OA=0.899, MacroF1=0.618, selected=25


,block_size_km,n_folds,n_selected_features_mean,OA_mean,OA_sd,Kappa_mean,Kappa_sd,MacroF1_10class_mean,MacroF1_10class_sd,CoffeeSubclassMacroF1_mean,CoffeeSubclassMacroF1_sd,CoffeeBinaryF1_mean,CoffeeBinaryF1_sd,CoffeeBinaryPrecision_mean,CoffeeBinaryPrecision_sd,CoffeeBinaryRecall_mean,CoffeeBinaryRecall_sd,nearest_train_test_distance_median_m_mean
0,10,5,25.0,0.761705,0.115818,0.728980,0.127635,0.718878,0.073002,0.384670,0.155955,0.866343,0.079059,0.877443,0.111787,0.864486,0.095930,3291.370578
1,15,5,25.0,0.744020,0.187281,0.703462,0.205856,0.639005,0.076710,0.174873,0.180600,0.658617,0.388249,0.643435,0.376094,0.678871,0.409563,5820.703668
2,20,5,25.0,0.745227,0.221040,0.705793,0.244453,0.606028,0.053202,0.151201,0.163890,0.479068,0.449279,0.490572,0.460720,0.470114,0.442092,5962.252014



## 10. Supplementary tables

The tables below are exported as `.csv`, `.md`, `.tex`, and one combined `.xlsx` workbook. The numbering can be adjusted to match the final manuscript order.


In [18]:

# ============================================================
# 10. Supplementary tables
# ============================================================

def mean_sd_text(mean_val, sd_val, decimals=3):
    if pd.isna(mean_val):
        return 'NA'
    if pd.isna(sd_val):
        return f'{mean_val:.{decimals}f}'
    return f'{mean_val:.{decimals}f} ± {sd_val:.{decimals}f}'


def save_table_bundle(df: pd.DataFrame, basename: str, table_dir: Path):
    table_dir.mkdir(parents=True, exist_ok=True)
    csv_path = table_dir / f'{basename}.csv'
    md_path = table_dir / f'{basename}.md'
    tex_path = table_dir / f'{basename}.tex'
    df.to_csv(csv_path, index=False)
    md_path.write_text(df.to_markdown(index=False), encoding='utf-8')
    tex_path.write_text(df.to_latex(index=False, escape=False), encoding='utf-8')
    return csv_path, md_path, tex_path

summary = cv_outputs['summary'].copy()
class_summary = cv_outputs['class_metrics_summary'].copy()
selection_frequency = cv_outputs['selection_frequency'].copy()
fold_metrics = cv_outputs['fold_metrics'].copy()
fold_class_counts = cv_outputs['fold_class_counts'].copy()

# Supplementary Table S3: overall spatial CV sensitivity
S3_spatial_cv = summary.copy().sort_values('block_size_km')
S3_spatial_cv['Selected predictors'] = S3_spatial_cv['n_selected_features_mean'].map(lambda v: f'{v:.1f}')
S3_spatial_cv['Overall accuracy'] = [mean_sd_text(m, s) for m, s in zip(S3_spatial_cv['OA_mean'], S3_spatial_cv['OA_sd'])]
S3_spatial_cv['Kappa'] = [mean_sd_text(m, s) for m, s in zip(S3_spatial_cv['Kappa_mean'], S3_spatial_cv['Kappa_sd'])]
S3_spatial_cv['Macro F1 (10 classes)'] = [mean_sd_text(m, s) for m, s in zip(S3_spatial_cv['MacroF1_10class_mean'], S3_spatial_cv['MacroF1_10class_sd'])]
S3_spatial_cv['Coffee-subclass macro F1'] = [mean_sd_text(m, s) for m, s in zip(S3_spatial_cv['CoffeeSubclassMacroF1_mean'], S3_spatial_cv['CoffeeSubclassMacroF1_sd'])]
S3_spatial_cv['Coffee binary F1'] = [mean_sd_text(m, s) for m, s in zip(S3_spatial_cv['CoffeeBinaryF1_mean'], S3_spatial_cv['CoffeeBinaryF1_sd'])]
S3_spatial_cv = S3_spatial_cv[[
    'block_size_km', 'n_folds', 'Selected predictors', 'Overall accuracy', 'Kappa',
    'Macro F1 (10 classes)', 'Coffee-subclass macro F1', 'Coffee binary F1',
    'nearest_train_test_distance_median_m_mean'
]].rename(columns={
    'block_size_km': 'Block size (km)',
    'n_folds': 'Folds',
    'nearest_train_test_distance_median_m_mean': 'Mean fold median nearest train-test distance (m)'
})

# Supplementary Table S4: class-wise F1
S4_base = class_summary[['block_size_km', 'class_id', 'class_name', 'f1_mean', 'f1_sd', 'producer_accuracy_mean', 'producer_accuracy_sd', 'user_accuracy_mean', 'user_accuracy_sd']].copy()
S4_base['block_label'] = S4_base['block_size_km'].astype(int).astype(str) + ' km'
S4_base['F1'] = [mean_sd_text(m, s) for m, s in zip(S4_base['f1_mean'], S4_base['f1_sd'])]
S4_class_f1 = (
    S4_base.pivot_table(index=['class_id', 'class_name'], columns='block_label', values='F1', aggfunc='first')
    .reset_index()
    .rename(columns={'class_id': 'Class ID', 'class_name': 'Class'})
)
ordered_block_cols = [f'{km} km' for km in BLOCK_SIZES_KM if f'{km} km' in S4_class_f1.columns]
S4_class_f1 = S4_class_f1[['Class ID', 'Class'] + ordered_block_cols]

# Supplementary Table S5: predictor selection frequency, wide format
S5 = selection_frequency.copy()
S5['block_label'] = S5['block_size_km'].astype(int).astype(str) + ' km'
S5_wide = S5.pivot_table(
    index=['feature', 'source_group', 'season_or_orbit', 'feature_family'],
    columns='block_label', values='selected_count_out_of_folds', fill_value=0, aggfunc='sum'
).reset_index()
for km in BLOCK_SIZES_KM:
    col = f'{km} km'
    if col not in S5_wide.columns:
        S5_wide[col] = 0
S5_wide['Total selected count'] = S5_wide[[f'{km} km' for km in BLOCK_SIZES_KM]].sum(axis=1)
S5_wide['Overall selection frequency (%)'] = 100 * S5_wide['Total selected count'] / (N_SPLITS * len(BLOCK_SIZES_KM))
S5_wide = S5_wide.sort_values(['Total selected count', 'feature'], ascending=[False, True]).reset_index(drop=True)
S5_wide = S5_wide.rename(columns={
    'feature': 'Predictor',
    'source_group': 'Source group',
    'season_or_orbit': 'Season/orbit',
    'feature_family': 'Feature family',
})

# Supplementary Table S6: fold-level diagnostics
S6_fold_diag = fold_metrics.copy().sort_values(['block_size_km', 'fold'])
S6_fold_diag = S6_fold_diag[[
    'block_size_km', 'fold', 'n_train_samples', 'n_test_samples', 'n_train_blocks', 'n_test_blocks',
    'n_candidate_features_after_fold_screen', 'n_selected_features', 'OA', 'Kappa', 'MacroF1_10class',
    'CoffeeSubclassMacroF1', 'CoffeeBinaryF1', 'oob_score',
    'nearest_train_test_distance_min_m', 'nearest_train_test_distance_p05_m',
    'nearest_train_test_distance_median_m', 'nearest_train_test_distance_mean_m'
]].rename(columns={'block_size_km': 'Block size (km)', 'fold': 'Fold'})

supp_tables = {
    'Supplementary_Table_S3_SpatialBlockCV_Sensitivity': S3_spatial_cv,
    'Supplementary_Table_S4_ClasswiseF1': S4_class_f1,
    'Detail_SpatialCV_PredictorSelectionFrequency': S5_wide,
    'Detail_SpatialCV_PerFoldDiagnostics': S6_fold_diag,
}

for name, table in supp_tables.items():
    save_table_bundle(table, name, TABLE_DIR)

# Excel workbook for convenience. CSV/MD/TEX are the primary manuscript-safe exports.
xlsx_path = TABLE_DIR / 'Supplementary_Tables_SpatialBlockCV_FULL_DataProcessing_Nature_v2.xlsx'
try:
    with pd.ExcelWriter(xlsx_path, engine='xlsxwriter') as writer:
        for name, table in supp_tables.items():
            sheet_name = name.replace('Supplementary_Table_', '')[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)
except Exception:
    with pd.ExcelWriter(xlsx_path) as writer:
        for name, table in supp_tables.items():
            sheet_name = name.replace('Supplementary_Table_', '')[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

S3_spatial_cv


,Block size (km),Folds,Selected predictors,Overall accuracy,Kappa,Macro F1 (10 classes),Coffee-subclass macro F1,Coffee binary F1,Mean fold median nearest train-test distance (m)
0,10,5,25.0,0.762 ± 0.116,0.729 ± 0.128,0.719 ± 0.073,0.385 ± 0.156,0.866 ± 0.079,3291.370578
1,15,5,25.0,0.744 ± 0.187,0.703 ± 0.206,0.639 ± 0.077,0.175 ± 0.181,0.659 ± 0.388,5820.703668
2,20,5,25.0,0.745 ± 0.221,0.706 ± 0.244,0.606 ± 0.053,0.151 ± 0.164,0.479 ± 0.449,5962.252014



## 11. Supplementary figures

The figures are generated with simple typography, compact panels and no decorative chart effects. Outputs are saved as both `.png` and `.pdf`.


In [19]:

# ============================================================
# 11. Nature-style figures
# ============================================================

def panel_label(ax, label: str, x=-0.12, y=1.05):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top', ha='left')


def save_figure(fig, basename: str):
    png = FIG_DIR / f'{basename}.png'
    pdf = FIG_DIR / f'{basename}.pdf'
    fig.savefig(png, dpi=FIG_DPI)
    fig.savefig(pdf, dpi=FIG_DPI)
    return png, pdf

# ---------- Figure S2: spatial distribution and blocks ----------
primary_blocked = assign_spatial_blocks(df2, PRIMARY_BLOCK_SIZE_KM)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for cid, sub in primary_blocked.groupby('class_id'):
    axes[0].scatter(sub['lon'], sub['lat'], s=9, alpha=0.7, label=f'{cid}. {CLASS_NAMES[cid]}')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Reference samples')
axes[0].legend(frameon=False, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc='upper left')
panel_label(axes[0], 'a')

block_plot = primary_blocked.groupby('block_id').agg(lon=('lon','mean'), lat=('lat','mean'), n=('sample_uid','size'), n_classes=('class_id','nunique')).reset_index()
sc = axes[1].scatter(block_plot['lon'], block_plot['lat'], s=block_plot['n'] * 1.5, c=block_plot['n_classes'], alpha=0.8)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title(f'{PRIMARY_BLOCK_SIZE_KM}-km spatial blocks')
cbar = fig.colorbar(sc, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label('Classes per block')
panel_label(axes[1], 'b')
save_figure(fig, 'Supplementary_Figure_S2_Spatial_Samples_and_Blocks')
plt.show()

# ---------- Figure S3: spatial CV performance and class F1 ----------
summary = cv_outputs['summary'].copy().sort_values('block_size_km')
class_summary = cv_outputs['class_metrics_summary'].copy()
selection_frequency = cv_outputs['selection_frequency'].copy()
fold_metrics = cv_outputs['fold_metrics'].copy()

fig = plt.figure(figsize=(12.5, 8.6), constrained_layout=True)
gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1.05, 1.0], height_ratios=[1, 1])

axA = fig.add_subplot(gs[0, 0])
x = summary['block_size_km'].values
metric_specs = [
    ('OA_mean', 'OA_sd', 'Overall accuracy'),
    ('Kappa_mean', 'Kappa_sd', 'Kappa'),
    ('MacroF1_10class_mean', 'MacroF1_10class_sd', 'Macro F1'),
    ('CoffeeSubclassMacroF1_mean', 'CoffeeSubclassMacroF1_sd', 'Coffee subclass F1'),
]
for mean_col, sd_col, label in metric_specs:
    axA.errorbar(x, summary[mean_col], yerr=summary[sd_col], marker='o', linewidth=1.2, capsize=3, label=label)
axA.set_xticks(x)
axA.set_xticklabels([f'{int(v)} km' for v in x])
axA.set_ylim(0, 1.02)
axA.set_ylabel('Score')
axA.set_xlabel('Spatial block size')
axA.set_title('Overall performance')
axA.legend(frameon=False, loc='lower left')
axA.grid(axis='y', alpha=0.25)
panel_label(axA, 'a')

axB = fig.add_subplot(gs[:, 1])
heat = class_summary.copy()
heat['block_label'] = heat['block_size_km'].astype(int).astype(str) + ' km'
heatmap = heat.pivot_table(index='class_name', columns='block_label', values='f1_mean', aggfunc='first')
heatmap = heatmap.reindex([CLASS_NAMES[i] for i in range(1, EXPECTED_N_CLASSES + 1)])
heatmap = heatmap[[f'{km} km' for km in BLOCK_SIZES_KM]]
mat = heatmap.values.astype(float)
im = axB.imshow(mat, aspect='auto', vmin=0, vmax=1)
axB.set_xticks(np.arange(mat.shape[1]))
axB.set_xticklabels(heatmap.columns.tolist())
axB.set_yticks(np.arange(mat.shape[0]))
axB.set_yticklabels(heatmap.index.tolist())
axB.set_title('Class-wise F1')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        axB.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=8, color='white' if mat[i,j] < 0.45 else 'black')
cbar = fig.colorbar(im, ax=axB, fraction=0.046, pad=0.04)
cbar.set_label('Mean F1')
panel_label(axB, 'b')

axC = fig.add_subplot(gs[1, 0])
sel = selection_frequency.copy()
sel_total = (
    sel.groupby(['feature', 'source_group'], as_index=False)['selected_count_out_of_folds'].sum()
    .assign(selection_frequency_pct=lambda z: 100 * z['selected_count_out_of_folds'] / (N_SPLITS * len(BLOCK_SIZES_KM)))
    .sort_values('selected_count_out_of_folds', ascending=False)
    .head(TOP_N_FEATURES_FIG)
    .sort_values('selected_count_out_of_folds')
)
axC.barh(sel_total['feature'], sel_total['selection_frequency_pct'])
axC.set_xlabel('Selection frequency across all spatial folds (%)')
axC.set_title(f'Top {TOP_N_FEATURES_FIG} selected predictors')
axC.set_xlim(0, 100)
panel_label(axC, 'c')

save_figure(fig, 'Supplementary_Figure_S3_SpatialBlockCV_Performance')
plt.show()



C:\Users\Owner\AppData\Local\Temp\ipykernel_5392\2030589382.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Owner\AppData\Local\Temp\ipykernel_5392\2030589382.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## 12. Manuscript-ready captions and output manifest

The next cell writes a concise manifest and suggested captions that can be pasted into the supplementary material.


In [20]:

# ============================================================
# 12. Manuscript notes and manifest
# ============================================================
manifest_rows = []
for folder in [AUDIT_DIR, CV_DIR, TABLE_DIR, FIG_DIR, NOTE_DIR]:
    for path in sorted(folder.glob('*')):
        if path.is_file():
            manifest_rows.append({'folder': folder.name, 'file': path.name, 'path': str(path), 'size_kb': round(path.stat().st_size / 1024, 1)})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_ROOT / 'OUTPUT_MANIFEST.csv', index=False)
manifest.head(20)


,folder,file,path,size_kb
0,supplementary,.gitkeep,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.0
1,supplementary,audit_summary_metrics.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.9
2,supplementary,class_distribution_train_validation.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.3
3,supplementary,class_metrics_by_fold.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,9.7
4,supplementary,class_metrics_summary.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,4.4
5,supplementary,classification_report_python_rf.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,1.0
6,supplementary,confusion_matrices_by_fold.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,5.1
7,supplementary,confusion_matrix_counts.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.3
8,supplementary,district_matching.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.6
9,supplementary,DistrictVsClassArea_Audit_20260715.csv,D:\2024_PhD_Research\Chap2_Mapping\coffeemap-d...,0.4



## 13. Final checks

Run this cell at the end. It checks whether the expected core outputs exist.


In [21]:

# ============================================================
# 13. Final output checks
# ============================================================
required_outputs = [
    CV_DIR / 'summary.csv',
    CV_DIR / 'fold_metrics.csv',
    CV_DIR / 'predictions.csv',
    TABLE_DIR / 'Supplementary_Table_S3_SpatialBlockCV_Sensitivity.csv',
    TABLE_DIR / 'Supplementary_Table_S4_ClasswiseF1.csv',
    TABLE_DIR / 'Detail_SpatialCV_PredictorSelectionFrequency.csv',
    FIG_DIR / 'Supplementary_Figure_S3_SpatialBlockCV_Performance.png',
    OUTPUT_ROOT / 'OUTPUT_MANIFEST.csv',
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError('Missing expected outputs:\n' + '\n'.join(missing))
print('All required core outputs exist.')
print(f'Output root: {OUTPUT_ROOT}')


All required core outputs exist.
Output root: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary


In [22]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)


,folder,file,size_kb
176,supplementary,S06_block_summary_20km.csv,3.3
177,supplementary,selected_features_by_fold.csv,23.4
178,supplementary,selected_features_used_by_RF_SHAP.csv,0.3
179,supplementary,selection_frequency.csv,9.3
180,supplementary,shap_class_10_importance.csv,2.0
181,supplementary,shap_class_1_importance.csv,2.1
182,supplementary,shap_class_2_importance.csv,2.3
183,supplementary,shap_class_3_importance.csv,2.3
184,supplementary,shap_class_4_importance.csv,2.0
185,supplementary,shap_class_5_importance.csv,2.3


Manifest saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\Manifest_mmlab-coffeemap-daklak.csv
